In [92]:
import pandas as pd
from bs4 import BeautifulSoup
import re
import datetime as date
import requests


URL = URL = "https://www.ccny.cuny.edu/registrar/fall"

In [93]:
response = requests.get(URL, headers = {"user": "Chrome (compatible; ccny-calendar-scraper)"}, timeout = 60, )
response.raise_for_status() #Creating a get request from the CCNY Website
html = response.text
response.status_code


200

In [94]:
soup = BeautifulSoup(html, "html.parser")

main = soup.find("main")
if main:
    text = main.get_text("\n", strip=True)
else:
    text = soup.get_text("\n", strip=True)

start_marker = "Fall 2021 Academic Calendar"
start_idx = text.find(start_marker)

if start_idx != -1:
    calendar_text = text[start_idx:]
else:
    calendar_text=text

In [95]:
MONTHS = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]


def parse_events(lines):
    events = []
    i =0

    while i < len(lines):
        line = lines[i].strip()

        parts= line.split()
        if len(parts) >= 2 and parts[0] in MONTHS and parts[1].replace(",","").isdigit():
            month = parts[0]
            day = int(parts[1].replace(",", ""))
            if month == "January":
                year = 2022
            else:
                year = 2021
            dow = ""
            if i +1 <len(lines):
                dow=lines[i+1].strip()

            j = i +2
            desc_lines = []
            
            while j < len(lines):
                next_line = lines[j].strip()
                parts_next = next_line.split()

                if (
                    len(parts_next) >= 2
                    and parts_next[0] in MONTHS
                    and parts_next[1].replace(",", "").isdigit()
                ):
                    break

                desc_lines.append(next_line)
                j+=1
            text = " ".join(desc_lines).strip()

            if "Last Updated:" in text:
                text = text.split("Last Updated:")[0].strip()
                
            dt=pd.Timestamp(year=year, month=MONTHS.index(month) + 1, day=day).date()

            events.append({"date": dt,
                           "dow": dow,
                           "text": text})
            i=j
        else:
            i+=1
    return events

lines = calendar_text.splitlines()
events = parse_events(lines)



In [96]:
df= pd.DataFrame(events)

df["date"] = pd.to_datetime(df["date"]).dt.date
df = df.drop_duplicates(subset=["date", "text"]).sort_values("date")
df= df.set_index("date")
df.index.name= None

df["dow"] = pd.to_datetime(df.index).day_name()
df=df[["dow", "text"]]

pd.set_option("display.expand_frame_repr", False)
pd.set_option("display.max_colwidth", None)
df.head(40)

,dow,text
2021-08-01,Sunday,Application for degree for January and February 2022 begins
2021-08-18,Wednesday,Last day to apply for Study Abroad
2021-08-24,Tuesday,Last day of Registration; Last day to file ePermit for the Fall 2021; Last day to drop classes for 100% tuition refund;
2021-08-25,Wednesday,Start of Fall Term; Classes begin; Initial Registration Appeals begin;
2021-08-25,Wednesday,Change of program period; late fees apply
2021-08-26,Thursday,Last day for Independent Study
2021-08-28,Saturday,First day of Saturday Classes
2021-08-31,Tuesday,Last day to add a class to an existing enrollment; Last day for 75% tuition refund; Financial Aid Certification Enrollment Status date; Last day to apply for Audit option; Last day to drop without the grade of 'WD'; Initial Registration Appeals end;
2021-09-01,Wednesday,Verification of Enrollment rosters available to faculty; Course Withdrawal drop period begins (A grade of 'WD' is assigned to students who officially drop a class);
2021-09-03,Friday,No classes scheduled
